In [1]:
# Importing standard libraries
import os
import gc
import sys
import warnings

# Importing data manipulation and scientific computation libraries
import pandas as pd
import numpy as np
import scanpy as sc
import anndata
import scipy.sparse as sparse

# Importing machine learning libraries
import torch
import umap

# Importing plotting libraries
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
from matplotlib.lines import Line2D
import skimage

# Importing scvi
import scvi
from scvi.data import synthetic_iid
from scvi.dataloaders import AnnDataLoader
from scvi.model import CondSCVI, DestVI

# # Importing cell2location
# import cell2location
# from cell2location import run_colocation
# from cell2location.models import Cell2location, RegressionModel

# # Importing tangram 
# import tangram as tg

# Importing pathlib for filesystem paths management
import pathlib

# Configure scanpy and suppress warnings
sc.logging.print_header()
warnings.filterwarnings('ignore')
import torch
torch.set_float32_matmul_precision('high')  # or 'high' for better performance with a potential precision trade-off

Global seed set to 0
/opt/conda/lib/python3.8/site-packages/flax/core/frozen_dict.py:169: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(


scanpy==1.9.2 anndata==0.8.0 umap==0.5.3 numpy==1.22.4 scipy==1.9.1 pandas==1.5.1 scikit-learn==1.2.1 statsmodels==0.13.5 python-igraph==0.10.4 pynndescent==0.5.8


In [ ]:
T_names = ["T808", "T809", "T996", "T814","T928"]

for profile in T_names:
    
    scdir = "/data/work/Layer division/00_Data/snRNA_downsample.h5ad"
    stdir = "/data/work/Layer division/00_Data/"+profile+"_Spatial-id.h5ad"
    key = 'subclass.v4'
    adata_sc= sc.read_h5ad(scdir)
    adata_st= sc.read_h5ad(stdir)

    sc.pp.filter_genes(adata_st, min_cells=10) 
    sc.pp.filter_genes(adata_sc, min_cells=10) # 每一个基因至少在10个细胞中表达

    adata_sc.layers["counts"]=adata_sc.X.copy()
    adata_st.layers["counts"]=adata_st.X.copy()

    Gnumber = 5000###基因的可以有多种选择，只需要和单细胞的数据做交集即可

    sc.pp.highly_variable_genes(adata_sc,n_top_genes=Gnumber,layer="counts",subset=True,flavor='seurat_v3') 
    sc.pp.normalize_total(adata_sc, target_sum=10e4) 
    sc.pp.log1p(adata_sc)

    adata_st.raw=adata_st
    sc.pp.normalize_total(adata_st, target_sum=10e4) 
    sc.pp.log1p(adata_st) 

    intersect = np.intersect1d(adata_sc.var_names,adata_st.var_names)####共同的gene
    adata_st=adata_st[:,intersect].copy()
    adata_sc=adata_sc[:,intersect].copy()

    len(intersect)

    CondSCVI.setup_anndata(adata_sc,layer="counts",labels_key=key)#")#需要修改
    sc_model=CondSCVI(adata_sc,weight_obs=True)
    sc_model.train(max_epochs=300)##训练周期太少可能导致好多细胞类型出不来


    DestVI.setup_anndata(adata_st,layer="counts")
    st_model=DestVI.from_rna_model(adata_st,sc_model)
    st_model.train(max_epochs=300)###训练周期太少可能导致好多细胞类型出不来

    # st_model.history["elbo_train"].plot()
    adata_st.obsm["proportions"]=st_model.get_proportions()
    adata_st.obsm["proportions"].to_csv("/data/work/Layer division/01_Result/Destvi_"+ profile +".cellbin_pret.csv")
    st_model.get_proportions()


No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 205/300:  68%|██████▊   | 204/300 [5:09:08<3:12:23, 120.25s/it, loss=2.64e+04, v_num=1]